# Retail Sales Performance Analysis — Live Demo

End-to-end walkthrough: generate raw data → clean it → run SQL analysis → visualize.

**Run this notebook top to bottom** (Runtime → Run all, in Colab) to see the full pipeline execute and produce real charts.

Repo: https://github.com/maditapumadhan-pixel/retail-sales-analysis


## 1. Setup — clone repo & install dependencies

In [ ]:
!git clone https://github.com/maditapumadhan-pixel/retail-sales-analysis.git
%cd retail-sales-analysis
!pip install -q pandas numpy matplotlib seaborn


## 2. Generate raw data
Creates a realistic 6,000+ row retail transactions dataset (with intentional data quality issues to clean later).

In [ ]:
!python scripts/generate_data.py

In [ ]:
import pandas as pd
raw = pd.read_csv("data/retail_sales_raw.csv")
print("Shape:", raw.shape)
raw.head()


## 3. Clean the data
Removes duplicates, standardizes text, handles missing values, adds derived columns.

In [ ]:
!python scripts/clean_data.py

In [ ]:
clean = pd.read_csv("data/retail_sales_clean.csv")
print("Cleaned shape:", clean.shape)
print("Duplicates remaining:", clean.duplicated().sum())
clean.head()


## 4. SQL Analysis
Load cleaned data into SQLite and run business-question queries (aggregations + window functions).

In [ ]:
import sqlite3

conn = sqlite3.connect("data/retail_sales.db")
clean.to_sql("sales", conn, if_exists="replace", index=False)

query = '''
SELECT region,
       ROUND(SUM(revenue), 2) AS total_revenue,
       ROUND(SUM(profit) * 100.0 / SUM(revenue), 2) AS profit_margin_pct
FROM sales
GROUP BY region
ORDER BY total_revenue DESC;
'''
pd.read_sql(query, conn)


In [ ]:
query_top_products = '''
SELECT product, category, SUM(quantity) AS units_sold, ROUND(SUM(revenue),2) AS total_revenue
FROM sales
GROUP BY product, category
ORDER BY total_revenue DESC
LIMIT 5;
'''
pd.read_sql(query_top_products, conn)


In [ ]:
query_rank = '''
SELECT category,
       ROUND(SUM(revenue),2) AS total_revenue,
       ROUND(SUM(profit),2) AS total_profit,
       RANK() OVER (ORDER BY SUM(profit) DESC) AS profit_rank
FROM sales
GROUP BY category;
'''
pd.read_sql(query_rank, conn)


## 5. EDA & Visualization
Generates 5 charts summarizing revenue, trends, margins, and correlations.

In [ ]:
!python scripts/eda_visualize.py

In [ ]:
from PIL import Image
display(Image.open("output/revenue_by_category.png"))


In [ ]:
display(Image.open("output/monthly_revenue_trend.png"))


In [ ]:
display(Image.open("output/profit_margin_by_region.png"))


In [ ]:
display(Image.open("output/channel_comparison.png"))


In [ ]:
display(Image.open("output/correlation_heatmap.png"))


## Key Findings

- Overall profit margin across all orders: **~31%**
- **Electronics** is the top revenue-generating category
- **South** region delivers the strongest profit margin
- Orders with 15%+ discounts frequently fall below a 20% margin threshold — flagged for pricing review

---
Author: Madhan Simha M · [GitHub](https://github.com/maditapumadhan-pixel) · [Repo](https://github.com/maditapumadhan-pixel/retail-sales-analysis)
